# Context-Aware Digital Twin: Fine-Tuned Llama 3.1 with RAG Contextual Memory

A "Digital Twin" that is aware of it's context and is capable of mimicking a specific person's conversational style while retaining a long-term memory of past interactions.

## 🚀 The Challenge
Standard LLMs can either be fine-tuned on a person's style (Static) or given a few documents to read (RAG). This project combines both:
1. **Style:** Fine-tuned Llama 3.1 (8B) using Unsloth to capture Roman Urdu/English "code-switching" and slang.
2. **Memory:** A ChromaDB vector store that indexes over 51420 messages to provide factual recall (inside jokes, nicknames, preferences).

## 🛠️ Technical Stack
- **Base Model:** Meta Llama 3.1 Quantized to 4 bits
- **Fine-Tuning:** Unsloth (PEFT/LoRA) for memory-efficient training.
- **Vector Database:** ChromaDB
- **Embeddings:** `all-MiniLM-L6-v2` (Sentence-Transformers)
- **Data Pipeline:** Custom Python regex scripts for cleaning WhatsApp export artifacts (iOS hidden characters, legal disclaimers, and multi-line message merging).

## 🧠 Key Features
- **Semantic Filtering:** Custom inference-time logic to filter out system "junk" (e.g., privacy notices) from retrieved context.
- **Dynamic Entity Recognition:** Bridging the gap between RAG retrieval and LLM logic to recognize user-specific identifiers (e.g., "AYK").
- **Hybrid Inference:** A response loop that prioritizes retrieved "memories" while maintaining the fine-tuned persona's tone.

## 📊 Dataset Cleaning
A significant portion of this project involved a custom preprocessing pipeline to convert raw `_chat.txt` exports into a structured JSONL format, handling:
- Multi-line message reconstruction.
- User-consecutive message merging.
- HTML tag stripping and invisible Unicode character removal.

In [ ]:
#imp installations
#pip install -U bitsandbytes
#pip install unsloth "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
#pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
#!pip install chromadb sentence-transformers

In [ ]:
#import necessary libraries
import re
import json
from datasets import load_dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TextStreamer
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
#data ingestion and cleaning block

INPUT_FILE = "_chat.txt"
OUTPUT_FILE = "training_data.jsonl"
FRIEND_NAME = "<Clone_Name>" #insert name as appears in chat
YOUR_NAME = "<Insert_Name>"

def clean_text_content(text):
    """Deep cleans messages of legal junk, system tags, and HTML."""
    #remove ios hidden markers
    text = text.replace('\u200e', '').replace('\u202f', ' ')

    #strip html tags
    text = re.sub(r'<.*?>', '', text)

    #remove junk
    junk_patterns = [
        r"Messages and calls are end-to-end encrypted",
        r"STOP to stop receiving messages",
        r"HELP to learn more",
        r"DELETE to delete all saved information",
        r"WhatsApp’s Privacy Policy",
        r"https://whatsapp.com/legal",
        r"pinned a message",
        r"changed the group description",
        r"omitted"
    ]

    for pattern in junk_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return None #mark for deletion

    return text.strip()

def process_chat(file_path):
    pattern = re.compile(r'^\[(\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}:\d{2}\s[APM]{2})\]\s([^:]+):\s(.*)') #regex based cleanup

    messages = []
    current_msg = None

    with open(file_path, 'r', encoding='utf-8-sig') as f:
        for line in f:
            cleaned_line = clean_text_content(line)
            if cleaned_line is None or not cleaned_line:
                continue

            match = pattern.match(cleaned_line)
            if match:
                if current_msg:
                    messages.append(current_msg)

                _, sender, text = match.groups()

                final_text = clean_text_content(text)
                if final_text:
                    current_msg = {"sender": sender.strip(), "text": final_text}
                else:
                    current_msg = None
            else:
                #multi line messages handling
                if current_msg:
                    current_msg["text"] += " " + cleaned_line

        if current_msg:
            messages.append(current_msg)
    return messages


print("Deep cleaning chat")
all_messages = process_chat(INPUT_FILE)

#merge consecutive messages by one person
merged = []
if all_messages:
    curr = all_messages[0]
    for i in range(1, len(all_messages)):
        if all_messages[i]['sender'] == curr['sender']:
            curr['text'] += " " + all_messages[i]['text']
        else:
            merged.append(curr)
            curr = all_messages[i]
    merged.append(curr)

#create text pairs
pairs = []
for i in range(len(merged) - 1):
    if merged[i]['sender'] == YOUR_NAME and merged[i+1]['sender'] == FRIEND_NAME:
        #final check for null strings
        if len(merged[i+1]['text']) > 1:
            pairs.append({
                "instruction": f"Respond as {FRIEND_NAME}",
                "input": merged[i]['text'],
                "output": merged[i+1]['text']
            })

#save
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for p in pairs:
        f.write(json.dumps(p) + '\n')

print(f"Cleaned {len(all_messages)} messages.")
print(f"Created {len(pairs)} high-quality training pairs.")
print(f"'merged' list is now ready for RAG indexing.")

In [ ]:
#LLM config

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

#LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

In [ ]:
#generate data mappings with prompt

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

dataset = load_dataset("json", data_files={"train": "training_data.jsonl"}, split="train")

#fix format
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

print("dataset is mapped.")

In [ ]:
#LLM training

#initialize the trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
#inference check 1

FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Respond as cloned friend",
        "Oye, did you see that video I sent?",
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64)
tokenizer.batch_decode(outputs)

In [ ]:
#inference check 2
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Respond as cloned friend",
        "what's ur coffee order",
        "",
    )
], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 64,
                   temperature = 0.8,
                   top_p = 0.9)

In [ ]:
#mount drive and save fine-tuned weights

from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/model_name")
tokenizer.save_pretrained("/content/drive/MyDrive/model_name")

In [ ]:
#RAG setup

#initialize model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

#chromaDB setup
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection(name="whatsapp_memory")
except:
    pass
collection = chroma_client.create_collection(name="whatsapp_memory")

source_data = merged if 'merged' in globals() else all_messages

documents = [msg['text'] for msg in source_data]
metadatas = [{"sender": msg['sender']} for msg in source_data]
ids = [str(i) for i in range(len(source_data))]

#insertion via batch processing
batch_size = 5000
for i in range(0, len(documents), batch_size):
    print(f"Adding batch {i//batch_size + 1}...")

    batch_docs = documents[i : i + batch_size]
    batch_metas = metadatas[i : i + batch_size]
    batch_ids = ids[i : i + batch_size]

    #generate embeddings for this batch
    batch_embeddings = embed_model.encode(batch_docs).tolist()

    collection.add(
        documents=batch_docs,
        metadatas=batch_metas,
        ids=batch_ids,
        embeddings=batch_embeddings
    )

print(f"Memory Bank created with {len(documents)} messages.")

In [ ]:
#output

def ask_final(question):
    query_embedding = embed_model.encode([question]).tolist()                   #filter relevant docs
    results = collection.query(query_embeddings=query_embedding, n_results=5)   #reduce noise

    #filter at RAG level by dist for better results
    valid_memories = []
    for doc, distance in zip(results['documents'][0], results['distances'][0]):
        if distance < 1.1:
            valid_memories.append(doc)

    #main prompt
    if valid_memories:
        context = "\n".join([f"- {m}" for m in valid_memories])
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are <Clone_Name>. You are talking to your friend Abdullah (AYK).
Answer ONLY using the provided past chat memories.
If the answer isn't there, say "I don't remember" or "No idea lol".

### Past Chat Memories:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
    else:  #fallback prompt if no retrieval from RAG, output based on persona picked up by LLM
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are <Clone_Name>. Answer Abdullah (AYK) naturally in your style.
There is no specific memory for this, so just be yourself.<|eot_id|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    #stabilize inference
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.4,
        repetition_penalty=1.2,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return response.split("assistant")[-1].strip()


print(f"Output: {ask_final('What is your fav perfume?')}")